# FPL AI Predictor — Historical Dataset

## 1. Load historical ML dataset

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from fpl_predictor.validation import audit_feature_leakage

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATASET_PATH = PROJECT_ROOT / 'data' / 'historical' / 'ml' / 'player_gameweek_dataset.csv'
dataset = pd.read_csv(DATASET_PATH, low_memory=False)

## 2. Dataset dimensions

In [ ]:
print(f'Rows: {len(dataset):,}')
print(f'Columns: {len(dataset.columns):,}')
print(f'Player-seasons: {dataset.season_player_id.nunique():,}')

## 3. Seasons available

In [ ]:
display(dataset.groupby('season').agg(rows=('gw', 'size'), players=('season_player_id', 'nunique'), gameweeks=('gw', 'nunique')))

## 4. Missing-data inspection

In [ ]:
missingness = dataset.isna().mean().sort_values(ascending=False)
display(missingness.head(25).to_frame('missing_fraction'))

## 5. Target-points distribution

In [ ]:
dataset.target_points.hist(bins=35, figsize=(9, 4))
plt.title('Actual FPL points per player-Gameweek')
plt.xlabel('target_points')
plt.show()

## 6. Rolling-form examples

In [ ]:
example_id = dataset.loc[dataset.minutes_last_5.notna(), 'season_player_id'].iloc[0]
display(dataset.query('season_player_id == @example_id')[['season', 'gw', 'player_name', 'points_last_3', 'xGI_last_3', 'target_points']].head(12))

## 7. Double Gameweek examples

In [ ]:
display(dataset.query('fixture_count > 1')[['season', 'gw', 'player_name', 'fixture_count', 'home_fixture_count', 'away_fixture_count', 'avg_fixture_difficulty', 'target_points']].head(15))

## 8. Blank Gameweek examples

In [ ]:
display(dataset.query('did_not_play_because_team_blank == True')[['season', 'gw', 'player_name', 'team', 'fixture_count', 'target_points']].head(15))

## 9. Feature correlations

In [ ]:
candidate_features = ['points_last_3', 'points_last_5', 'minutes_last_3', 'xGI_last_3', 'team_goals_for_last_3', 'opponent_goals_against_last_3', 'target_points']
display(dataset[candidate_features].corr(method='spearman').round(3))

## 10. Baseline prediction check

In [ ]:
baseline_rows = dataset.dropna(subset=['predicted_points_baseline', 'target_points'])
baseline_mae = (baseline_rows.target_points - baseline_rows.predicted_points_baseline).abs().mean()
print(f'Prior-3-GW mean baseline MAE: {baseline_mae:.3f}')

## 11. Leakage sanity checks

In [ ]:
targets = {'target_points', 'target_minutes'}
identifiers = {'season', 'gw', 'season_player_id', 'player_id', 'player_name', 'team', 'position'}
features = [column for column in dataset.columns if column not in targets | identifiers]
print('Suspicious unlagged outcomes:', audit_feature_leakage(features))
rolling_columns = [column for column in features if '_last_' in column]
print('GW1 rolling values present:', int(dataset.loc[dataset.gw.eq(1), rolling_columns].notna().sum().sum()))
print('Duplicate player-season-GWs:', int(dataset.duplicated(['season', 'season_player_id', 'gw']).sum()))

## 12. Initial observations

Per-90 rolling values are intentionally missing when a prior window contains zero minutes. Blank rows remain explicit and Double Gameweeks retain aggregate fixture context. Correlations and the prior-three-Gameweek baseline are descriptive checks only; no Phase 3 model is trained here.